**2. Análisis Descriptivo**

Este script realiza el análisis de los datos previamente limpiados, unificando los archivos de distintas áreas del conocimiento en un solo conjunto. Calcula métricas agregadas como totales y promedios de vistas, likes, comentarios y duración por área.

Posteriormente, normaliza estas métricas utilizando z-score para permitir comparaciones más justas entre áreas. Finalmente, genera tablas resumen y visualizaciones gráficas que muestran la distribución y comparación de las métricas, exportando todos los resultados a archivos para su interpretación.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore
import numpy as np

# 1. CONFIGURACIÓN
DATA_DIR = "."
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

files = [
    "biología.csv",
    "ciencias_sociales.csv",
    "física.csv",
    "ingeniería.csv",
    "matemáticas.csv",
    "química.csv",
    "salud_medicina.csv",
    "tecnología_ia.csv",
]

METRICAS = ["vistas", "likes", "comentarios", "duracion_segundos"]

# 2. FUNCIONES AUXILIARES
def leer_csv(path):
    df = pd.read_csv(path, dtype=str)
    for col in METRICAS + ["suscriptores_canal"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].str.replace(",", "").str.strip(), errors="coerce")
        else:
            df[col] = pd.NA
    if "fecha_publicacion" in df.columns:
        df["fecha_publicacion"] = pd.to_datetime(df["fecha_publicacion"], errors="coerce")
    if "descripcion" in df.columns:
        df["descripcion"] = df["descripcion"].fillna("").astype(str)
    else:
        df["descripcion"] = ""
    return df

def nombre_area_desde_archivo(fname):
    return os.path.splitext(os.path.basename(fname))[0]

def guardar_fig(fig, fname):
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, fname))
    plt.close(fig)

# 3. CARGA Y UNIFICACIÓN DE DATOS
frames = []
for fname in files:
    path = os.path.join(DATA_DIR, fname)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"No encontrado: {path}")
    df = leer_csv(path)
    df["area"] = nombre_area_desde_archivo(fname)
    frames.append(df)

df_all = pd.concat(frames, ignore_index=True)

# 4. RESUMEN POR ÁREA
group = df_all.groupby("area", dropna=False)
summary = pd.DataFrame(index=sorted(group.groups.keys()))

summary["n_videos"] = group.size()

for m in METRICAS:
    summary[f"{m}_total"] = group[m].sum(min_count=1)
    summary[f"{m}_mean"] = group[m].mean()

total_cols = [c for c in summary.columns if c.endswith("_total")]
summary[total_cols] = summary[total_cols].fillna(0)

# 5. NORMALIZACIÓN (Z-SCORE)
for m in METRICAS:
    summary[f"{m}_norm"] = zscore(summary[f"{m}_mean"].fillna(0))

cols_order = ["n_videos"] + [c for c in summary.columns if c not in ["n_videos"]]
summary = summary[cols_order]

summary_reset = summary.reset_index().rename(columns={"index": "area"})
summary_reset.to_csv(os.path.join(OUTPUT_DIR, "comparativa_resumen_por_area.csv"), index=False)
summary.to_excel(os.path.join(OUTPUT_DIR, "comparativa_resumen_por_area.xlsx"))
summary_reset.to_csv(os.path.join(OUTPUT_DIR, "tabla_comparativa_ancha.csv"), index=False)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Resumen por área:")
print(summary)

# 6. VISUALIZACIÓN
for m in METRICAS:
    total_col = f"{m}_total"
    mean_col = f"{m}_mean"
    norm_col = f"{m}_norm"

    fig, ax = plt.subplots()
    summary[total_col].sort_values(ascending=False).plot(kind="bar", ax=ax)
    ax.set_title(f"Total de {m} por área")
    ax.set_ylabel(total_col)
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    guardar_fig(fig, f"total_{m}.png")

    fig, ax = plt.subplots()
    summary[mean_col].sort_values(ascending=False).plot(kind="bar", ax=ax)
    ax.set_title(f"Promedio de {m} por área")
    ax.set_ylabel(mean_col)
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    guardar_fig(fig, f"promedio_{m}.png")

    fig, ax = plt.subplots()
    summary[norm_col].sort_values(ascending=False).plot(kind="bar", ax=ax, color="green")
    ax.set_title(f"Métrica normalizada (z-score) de {m} por área")
    ax.set_ylabel("Z-score")
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    guardar_fig(fig, f"normalizado_{m}.png")

totales_para_graf = summary[[f"{m}_total" for m in METRICAS]].copy()
totales_para_graf.columns = [m for m in METRICAS]

fig, ax = plt.subplots()
totales_para_graf.sort_values(by="vistas", ascending=False).plot(kind="bar", ax=ax)
ax.set_title("Comparativa de totales de métricas por área")
ax.set_ylabel("Total")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
guardar_fig(fig, "comparativa_totales_vistas_likes_comentarios_duracion.png")

metricas_norm = [f"{m}_norm" for m in METRICAS]

fig, ax = plt.subplots(figsize=(10, 6))
areas = summary.index
x = np.arange(len(areas))
width = 0.2

for i, m in enumerate(metricas_norm):
    ax.bar(x + i * width, summary[m], width=width, label=m.replace("_norm", ""))

ax.set_xticks(x + width * (len(metricas_norm) - 1) / 2)
ax.set_xticklabels(areas, rotation=30, ha="right")
ax.set_ylabel("Valor normalizado (z-score)")
ax.set_title("Comparativa de métricas normalizadas por área")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "comparativa_promedios_normalizados.png"))
plt.close(fig)

metricas_mean = [f"{m}_mean" for m in METRICAS]

fig, ax = plt.subplots(figsize=(10, 6))
areas = summary.index
x = np.arange(len(areas))
width = 0.2

for i, m in enumerate(metricas_mean):
    ax.bar(x + i * width, summary[m], width=width, label=m.replace("_mean", ""))

ax.set_xticks(x + width * (len(metricas_mean) - 1) / 2)
ax.set_xticklabels(areas, rotation=30, ha="right")
ax.set_ylabel("Promedio por video")
ax.set_title("Comparativa de promedios de métricas por área")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "comparativa_promedios_por_area.png"))
plt.close(fig)

# 7. EXPORTACIÓN
df_all.to_csv(os.path.join(OUTPUT_DIR, "dataset_concatenado_limpio.csv"), index=False)

print("\nArchivos guardados en:", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", f)


**3. Análisis de engagement**

Se calculan métricas de engagement para videos de distintas áreas del conocimiento, como interacción respecto a vistas, suscriptores del canal y duración del video. A partir de estos valores, obtiene promedios por área para comparar el nivel de interacción del público.

Además, genera visualizaciones que muestran estas métricas de forma individual y normalizada, permitiendo identificar qué áreas presentan mayor engagement en diferentes contextos.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# 1. CONFIGURACIÓN
archivos = {
    "Biología": "biología.csv",
    "Ciencias Sociales": "ciencias_sociales.csv",
    "Física": "física.csv",
    "Ingeniería": "ingeniería.csv",
    "Matemáticas": "matemáticas.csv",
    "Química": "química.csv",
    "Salud/Medicina": "salud_medicina.csv",
    "Tecnología/IA": "tecnología_ia.csv"
}

# 2. FUNCIONES AUXILIARES
def calcular_metricas(df):
    df = df.copy()

    df["vistas"] = df["vistas"].replace(0, pd.NA)
    df["suscriptores_canal"] = df["suscriptores_canal"].replace(0, pd.NA)
    df["duracion_segundos"] = df["duracion_segundos"].replace(0, pd.NA)

    df["Engagement Rate"] = (df["likes"] + df["comentarios"]) / df["vistas"]
    df["Engagement por Suscriptor"] = (df["likes"] + df["comentarios"]) / df["suscriptores_canal"]
    df["Engagement por Minuto"] = (df["likes"] + df["comentarios"]) / (df["duracion_segundos"] / 60)

    return df

# 3. CÁLCULO PRINCIPAL
resultados = []

for tema, archivo in archivos.items():
    if not os.path.exists(archivo):
        print(f"Advertencia: no se encontró {archivo}")
        continue

    df = pd.read_csv(archivo)
    df = calcular_metricas(df)

    promedios = {
        "Tema": tema,
        "Engagement Rate": df["Engagement Rate"].mean(),
        "Engagement por Suscriptor": df["Engagement por Suscriptor"].mean(),
        "Engagement por Minuto": df["Engagement por Minuto"].mean()
    }
    resultados.append(promedios)

df_promedios = pd.DataFrame(resultados)
df_promedios.to_csv("tabla_promedios_A_B_E.csv", index=False)
print("Tabla de promedios guardada: tabla_promedios_A_B_E.csv")

# 4. VISUALIZACIÓN
plt.figure(figsize=(10, 5))
plt.bar(df_promedios["Tema"], df_promedios["Engagement Rate"], color="steelblue")
plt.title("Promedio de Engagement Rate por Tema")
plt.ylabel("Promedio")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("grafica_engagement_rate.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 5))
plt.bar(df_promedios["Tema"], df_promedios["Engagement por Suscriptor"], color="orange")
plt.title("Promedio de Engagement por Suscriptor por Tema")
plt.ylabel("Promedio")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("grafica_engagement_por_suscriptor.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 5))
plt.bar(df_promedios["Tema"], df_promedios["Engagement por Minuto"], color="green")
plt.title("Promedio de Engagement por Minuto por Tema")
plt.ylabel("Promedio")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("grafica_engagement_por_minuto.png", dpi=300)
plt.close()

df_norm = df_promedios.copy()
cols = ["Engagement Rate", "Engagement por Suscriptor", "Engagement por Minuto"]
df_norm[cols] = df_norm[cols].div(df_norm[cols].max())

plt.figure(figsize=(12, 6))
bar_width = 0.25
x = range(len(df_norm))

plt.bar([i - bar_width for i in x], df_norm["Engagement Rate"], width=bar_width, label="Engagement Rate", color="steelblue")
plt.bar(x, df_norm["Engagement por Suscriptor"], width=bar_width, label="Engagement por Suscriptor", color="orange")
plt.bar([i + bar_width for i in x], df_norm["Engagement por Minuto"], width=bar_width, label="Engagement por Minuto", color="green")

plt.xticks(x, df_norm["Tema"], rotation=45, ha="right")
plt.title("Comparación normalizada de Engagement por Tema")
plt.ylabel("Valor normalizado (0–1)")
plt.legend()
plt.tight_layout()
plt.savefig("grafica_comparativa_normalizada.png", dpi=300)
plt.close()

print("Gráficas generadas y guardadas en el directorio actual.")


**4. Análisis de correlación entre variables**

Se combinan los datasets de distintas áreas del conocimiento en un solo conjunto de datos y calcula una métrica de engagement basada en la interacción de los usuarios. Posteriormente, analiza la relación entre variables como vistas, likes, comentarios y duración mediante matrices de correlación.

Finalmente, genera visualizaciones tipo heatmap y guarda tanto las gráficas como las matrices de correlación, permitiendo identificar patrones y relaciones entre las métricas a nivel global y por área.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CONFIGURACIÓN
FILES = {
    "biologia": "biología.csv",
    "ciencias_sociales": "ciencias_sociales.csv",
    "fisica": "fÍsica.csv",
    "ingenieria": "ingenierÍa.csv",
    "matematicas": "matemÁticas.csv",
    "quimica": "quÍmica.csv",
    "salud_medicina": "salud_medicina.csv",
    "tecnologia_ia": "tecnología_ia.csv",
}

DATA_DIR = "."
COMBINED_CSV = os.path.join(DATA_DIR, "todos_los_videos_combinado.csv")
COMBINED_WITH_ENGAGEMENT_CSV = os.path.join(DATA_DIR, "todos_los_videos_combinado_con_engagement.csv")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs_heatmaps")
os.makedirs(OUTPUT_DIR, exist_ok=True)

HEATMAP_VARS = ["vistas", "likes", "comentarios", "duracion_segundos", "engagement_rate"]

# 2. FUNCIONES AUXILIARES
def load_csv_safe(path):
    df = pd.read_csv(path, dtype=str)
    for col in ["vistas", "likes", "comentarios", "duracion_segundos", "suscriptores_canal"]:
        if col in df.columns:
            df[col] = df[col].str.replace(r"[^\d\.-]", "", regex=True).replace("", np.nan)
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def calcular_engagement(df):
    likes = df.get("likes")
    comentarios = df.get("comentarios")
    vistas = df.get("vistas")
    if likes is None: likes = pd.Series(np.nan, index=df.index)
    if comentarios is None: comentarios = pd.Series(np.nan, index=df.index)
    if vistas is None: vistas = pd.Series(np.nan, index=df.index)
    eng = (likes.fillna(0) + comentarios.fillna(0)) / vistas.replace({0: np.nan})
    df["engagement_rate"] = eng
    return df

def corr_and_heatmap(df, title, outpath, vars_list=HEATMAP_VARS, annot=True, figsize=(6, 5)):
    present_vars = [v for v in vars_list if v in df.columns]
    if len(present_vars) < 2:
        print(f"No hay suficientes columnas numéricas para {title}. Columnas presentes: {present_vars}")
        return None, None
    sub = df[present_vars].copy()
    sub = sub.apply(pd.to_numeric, errors="coerce")
    corr = sub.corr(method="pearson")
    plt.figure(figsize=figsize)
    sns.heatmap(
        corr, annot=annot, fmt=".2f", vmin=-1, vmax=1, square=True,
        linewidths=0.5, cbar_kws={"shrink": .8}
    )
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()
    return corr, present_vars

# 3. CARGA Y COMBINACIÓN DE DATOS
dfs = {}
for area, fname in FILES.items():
    path = os.path.join(DATA_DIR, fname)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    df_area = load_csv_safe(path)
    df_area["area_conocimiento"] = area
    dfs[area] = df_area

df_all = pd.concat(dfs.values(), ignore_index=True, sort=False)

df_all.to_csv(COMBINED_CSV, index=False)
print(f"Dataset combinado (sin engagement) guardado en: {COMBINED_CSV}")

# 4. CÁLCULO DE ENGAGEMENT Y EXPORTACIÓN
df_all = calcular_engagement(df_all)
df_all.to_csv(COMBINED_WITH_ENGAGEMENT_CSV, index=False)
print(f"Dataset combinado con columna engagement_rate guardado en: {COMBINED_WITH_ENGAGEMENT_CSV}")

# 5. MATRICES DE CORRELACIÓN Y HEATMAPS
global_corr_path = os.path.join(OUTPUT_DIR, "heatmap_global_todas_las_areas.png")
corr_global, vars_present_global = corr_and_heatmap(
    df_all,
    title="Correlación global - todas las áreas",
    outpath=global_corr_path
)
if corr_global is not None:
    print(f"Heatmap global guardado en: {global_corr_path}")

corrs_por_area = {}
for area, df_area in dfs.items():
    df_area = calcular_engagement(df_area)
    outpath = os.path.join(OUTPUT_DIR, f"heatmap_{area}.png")
    corr_area, vars_present = corr_and_heatmap(
        df_area,
        title=f"Correlación - {area}",
        outpath=outpath
    )
    corrs_por_area[area] = corr_area
    if corr_area is not None:
        print(f"Heatmap para '{area}' guardado en: {outpath}")

# 6. GUARDADO DE MATRICES DE CORRELACIÓN
if corr_global is not None:
    corr_global.to_csv(os.path.join(OUTPUT_DIR, "corr_matrix_global.csv"))
for area, corr_df in corrs_por_area.items():
    if corr_df is not None:
        corr_df.to_csv(os.path.join(OUTPUT_DIR, f"corr_matrix_{area}.csv"))

print("Análisis completado. Revisa la carpeta 'outputs_heatmaps' para los PNG y CSV de correlaciones.")


**5. Análisis temporal**

Se realiza un análisis temporal de las vistas de los videos por área del conocimiento, agrupando la información por mes a partir de la fecha de publicación. Calcula tanto el total como el promedio de vistas mensuales para cada área.

Además, genera gráficos individuales por área y visualizaciones globales comparativas, permitiendo observar la evolución y tendencias del consumo de contenido a lo largo del tiempo.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# 1. CONFIGURACIÓN
areas = [
    "biología",
    "ciencias_sociales",
    "física",
    "ingeniería",
    "matemáticas",
    "química",
    "salud_medicina",
    "tecnología_ia"
]

BASE_OUTPUT = "analisis_temporal_resultados"
os.makedirs(BASE_OUTPUT, exist_ok=True)

resumenes_globales = {}

# 2. PROCESAMIENTO POR ÁREA
for area in areas:

    fname = f"{area}.csv"
    df = pd.read_csv(fname)

    df["fecha_publicacion"] = pd.to_datetime(df["fecha_publicacion"], utc=True)
    df["año_mes"] = df["fecha_publicacion"].dt.to_period("M").dt.to_timestamp()

    resumen = df.groupby("año_mes")["vistas"].agg(
        vistas_total="sum",
        vistas_promedio="mean"
    ).reset_index()

    resumenes_globales[area] = resumen

    area_folder = os.path.join(BASE_OUTPUT, area)
    os.makedirs(area_folder, exist_ok=True)

    resumen.to_csv(os.path.join(area_folder, f"{area}_vistas_mensual.csv"), index=False)

    plt.figure(figsize=(10, 5))
    plt.plot(resumen["año_mes"], resumen["vistas_total"])
    plt.title(f"{area} — Total de vistas por mes")
    plt.xlabel("Mes")
    plt.ylabel("Total de vistas")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(area_folder, f"{area}_grafico_total_vistas.png"))
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.plot(resumen["año_mes"], resumen["vistas_promedio"])
    plt.title(f"{area} — Promedio de vistas por mes")
    plt.xlabel("Mes")
    plt.ylabel("Promedio de vistas")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(area_folder, f"{area}_grafico_promedio_vistas.png"))
    plt.close()

# 3. GRÁFICOS GLOBALES
carpeta_global = os.path.join(BASE_OUTPUT, "graficos_globales")
os.makedirs(carpeta_global, exist_ok=True)

plt.figure(figsize=(12, 6))
for area, df_res in resumenes_globales.items():
    plt.plot(df_res["año_mes"], df_res["vistas_total"], label=area)

plt.title("Todas las áreas — Total de vistas por mes")
plt.xlabel("Mes")
plt.ylabel("Total de vistas")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(carpeta_global, "total_vistas_todas_areas.png"))
plt.close()

plt.figure(figsize=(12, 6))
for area, df_res in resumenes_globales.items():
    plt.plot(df_res["año_mes"], df_res["vistas_promedio"], label=area)

plt.title("Todas las áreas — Promedio de vistas por mes")
plt.xlabel("Mes")
plt.ylabel("Promedio de vistas")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(carpeta_global, "promedio_vistas_todas_areas.png"))
plt.close()

print("Análisis temporal completado con gráficos globales.")


**6. Métrica Compuesta**

Se construye una métrica de popularidad para los videos combinando vistas, likes y comentarios mediante normalización (z-score). A partir de esto, calcula tanto una popularidad general como una versión ponderada que da mayor peso a la interacción de los usuarios.

Posteriormente, genera tablas y visualizaciones que permiten comparar la popularidad promedio entre áreas, analizar su distribución y destacar los videos más populares, facilitando la identificación de contenidos con mayor impacto.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# 1. CONFIGURACIÓN
ruta_datos = "./"
carpeta_export = "resultados_popularidad"
os.makedirs(carpeta_export, exist_ok=True)

archivos = {
    "Biología": "biología.csv",
    "Ciencias Sociales": "ciencias_sociales.csv",
    "Física": "física.csv",
    "Ingeniería": "ingeniería.csv",
    "Matemáticas": "matemáticas.csv",
    "Química": "química.csv",
    "Salud/Medicina": "salud_medicina.csv",
    "Tecnología/IA": "tecnología_ia.csv"
}

# 2. CARGA Y UNIFICACIÓN DE DATOS
dfs = []
for area, archivo in archivos.items():
    df_temp = pd.read_csv(os.path.join(ruta_datos, archivo))
    df_temp["area"] = area
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

# 3. NORMALIZACIÓN (Z-SCORE)
for col in ["vistas", "likes", "comentarios"]:
    df[f"{col}_norm"] = (df[col] - df[col].mean()) / df[col].std()

# 4. CÁLCULO DE MÉTRICAS COMPUESTAS
df["popularidad"] = (df["vistas_norm"] + df["likes_norm"] + df["comentarios_norm"]) / 3
df["popularidad_ponderada"] = (
    df["vistas_norm"] * 0.2 +
    df["likes_norm"] * 0.3 +
    df["comentarios_norm"] * 0.5
)

# 5. EXPORTACIÓN DE TABLA COMPLETA
df.to_csv(os.path.join(carpeta_export, "tabla_completa_con_popularidad.csv"), index=False)

# 6. PROMEDIOS POR ÁREA
tabla_promedios = df.groupby("area")[["popularidad", "popularidad_ponderada"]].mean()
tabla_promedios.to_csv(os.path.join(carpeta_export, "promedios_popularidad.csv"))

# 7. VISUALIZACIÓN DE PROMEDIOS
plt.figure(figsize=(10, 6))
plt.bar(tabla_promedios.index, tabla_promedios["popularidad"])
plt.xticks(rotation=45)
plt.title("Promedio de Popularidad por Área (Z-score)")
plt.ylabel("Popularidad Promedio")
plt.tight_layout()
plt.savefig(os.path.join(carpeta_export, "promedio_popularidad_por_area.png"))
plt.close()

plt.figure(figsize=(10, 6))
plt.bar(tabla_promedios.index, tabla_promedios["popularidad_ponderada"])
plt.xticks(rotation=45)
plt.title("Promedio de Popularidad Ponderada por Área (Z-score + Pesos)")
plt.ylabel("Popularidad Ponderada")
plt.tight_layout()
plt.savefig(os.path.join(carpeta_export, "promedio_popularidad_ponderada_por_area.png"))
plt.close()

# 8. DISTRIBUCIÓN DE POPULARIDAD
plt.figure(figsize=(10, 6))
plt.hist(df["popularidad"], bins=30)
plt.title("Distribución de Popularidad (Todos los Videos)")
plt.xlabel("Popularidad (Z-score)")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.savefig(os.path.join(carpeta_export, "distribucion_popularidad_total.png"))
plt.close()

for area in archivos.keys():
    plt.figure(figsize=(8, 5))
    subset = df[df["area"] == area]["popularidad"]
    plt.hist(subset, bins=30)
    plt.title(f"Distribución de Popularidad en {area}")
    plt.xlabel("Popularidad (Z-score)")
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.savefig(os.path.join(carpeta_export, f"histograma_popularidad_{area.replace('/', '-')}.png"))
    plt.close()

# 9. TOP 20 VIDEOS (POPULARIDAD VS. POPULARIDAD PONDERADA)
abrevs = {
    "Biología": "BIO",
    "Ciencias Sociales": "SOC",
    "Física": "FIS",
    "Ingeniería": "ING",
    "Matemáticas": "MAT",
    "Química": "QUI",
    "Salud/Medicina": "SAL",
    "Tecnología/IA": "TEC"
}

top20["titulo_area"] = top20.apply(lambda x: f"[{abrevs[x['area']]}] {x['titulo']}", axis=1)

plt.figure(figsize=(13, 10))

top20_sorted = top20.sort_values("popularidad", ascending=True)
pos = np.arange(len(top20_sorted))

plt.barh(pos - 0.2, top20_sorted["popularidad"], height=0.4, label="Popularidad Z-score")
plt.barh(pos + 0.2, top20_sorted["popularidad_ponderada"], height=0.4, label="Popularidad Ponderada")

plt.yticks(pos, top20_sorted["titulo_area"])
plt.xlabel("Nivel de Popularidad")
plt.title("Top 20 Videos Más Populares (Popularidad vs. Popularidad Ponderada)")
plt.legend()

for i, v in enumerate(top20_sorted["popularidad"]):
    plt.text(v, i - 0.2, f"{v:.2f}", va="center", ha="left", fontsize=8)

for i, v in enumerate(top20_sorted["popularidad_ponderada"]):
    plt.text(v, i + 0.2, f"{v:.2f}", va="center", ha="left", fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(carpeta_export, "grafica_top20_popularidad.png"))
plt.close()
